In [ ]:
# pip install cvxpy numpy
import numpy as np
import cvxpy as cp

def solve_markowitz(c, Sigma, gamma, solver="SCS"):
    """
    Solve: max c^T x  s.t. x^T Σ x ≤ γ, 1^T x ≤ 1, x ≥ 0.

    Parameters
    ----------
    c : (n,) array_like
        Expected returns vector.
    Sigma : (n,n) array_like
        (Semi)positive-definite covariance matrix.
    gamma : float
        Risk budget (must be ≥ 0).
    solver : str
        CVXPY solver name (e.g., "OSQP", "ECOS", "SCS").

    Returns
    -------
    x_opt : (n,) ndarray
        Optimal portfolio weights.
    obj_val : float
        Optimal objective value c^T x_opt.
    status : str
        CVXPY problem status.
    """
    c = np.asarray(c, dtype=float).reshape(-1)
    Sigma = np.asarray(Sigma, dtype=float)

    n = c.size
    assert Sigma.shape == (n, n), "Sigma must be n×n to match c."
 
    if gamma < 0:
        raise ValueError("gamma must be nonnegative.")

    # Symmetrize Σ (defensive) and ensure tiny numerical PSD with jitter if needed
    Sigma = 0.5 * (Sigma + Sigma.T)
    # Jitter only if numerically necessary
    eig_min = np.linalg.eigvalsh(Sigma).min()
    if eig_min < 0:
        Sigma = Sigma + (1e-10 - eig_min) * np.eye(n)

    x = cp.Variable(n, nonneg=True)

    constraints = [
        cp.quad_form(x, Sigma) <= gamma,
        cp.sum(x) <= 1.0
    ]

    # Maximize c^T x  <=>  Minimize -(c^T x)
    objective = cp.Maximize(c @ x)
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=solver, verbose=False)

    return (x.value if x.value is not None else None), prob.value, prob.status



In [ ]:
# --- Example usage ---
if __name__ == "__main__":
    np.random.seed(0)
    n = 6
    # Generate a random PSD covariance: A A^T + diag
    A = np.random.randn(n, n)
    Sigma = A @ A.T
    Sigma += 0.1 * np.eye(n)

    # Random expected returns
    c = np.random.uniform(0.02, 0.15, size=n)

    # Single solve
    gamma = 0.5
    x_star, ret_star, status = solve_markowitz(c, Sigma, gamma)
    print("Status:", status)
    print("Optimal return:", ret_star)
    print("Risk (x^T Σ x):", float(x_star @ Sigma @ x_star))
    print("Sum weights:", x_star.sum())
    print("x*:", np.round(x_star, 4))

In [ ]:
# pip install gurobipy numpy
import numpy as np
import gurobipy as gp
from gurobipy import GRB

def solve_bilevel_markowitz(c, Sigma, gamma, p_max=None, time_limit=None, verbose=True):
    """
    Solve:
        max_{p,y}  p^T y
        s.t. y in argmax (c - p)^T y
             s.t. y^T Σ y ≤ γ, 1^T y ≤ 1, y ≥ 0

    Implemented via KKT conditions (single-level nonconvex QCQP).
    Requires Gurobi with NonConvex=2.

    Parameters
    ----------
    c : (n,) array_like         follower's base utilities/returns
    Sigma : (n,n) array_like    PSD covariance matrix
    gamma : float               risk budget (≥ 0)
    p_max : float or (n,)       optional upper bound(s) on p to tighten/regularize
    time_limit : float          optional solver time limit in seconds
    verbose : bool              Gurobi output flag

    Returns
    -------
    result : dict with keys
        'p', 'y', 'lambda', 'mu', 'nu', 'obj', 'status'
    """
    c = np.asarray(c, dtype=float).reshape(-1)
    Sigma = np.asarray(Sigma, dtype=float)
    n = c.size
    assert Sigma.shape == (n, n), "Sigma must be n×n"

    # Symmetrize Σ for numerical stability
    Sigma = 0.5 * (Sigma + Sigma.T)

    m = gp.Model("bilevel_markowitz_kkt")
    m.Params.NonConvex = 2
    if time_limit is not None:
        m.Params.TimeLimit = float(time_limit)
    m.Params.OutputFlag = 1 if verbose else 0

    # Variables
    p = m.addMVar(n, lb=0.0, name="p")                 # price vector (bounded below by 0)
    y = m.addMVar(n, lb=0.0, name="y")                 # follower decision
    lam = m.addVar(lb=0.0, name="lambda")              # dual for risk constraint
    mu  = m.addVar(lb=0.0, name="mu")                  # dual for budget constraint
    nu  = m.addMVar(n, lb=0.0, name="nu")              # duals for y >= 0

    # Optional upper bounds on p (helps numerics & avoids pathological scaling)
    if p_max is not None:
        if np.isscalar(p_max):
            m.addConstr(p <= float(p_max))
        else:
            pmax = np.asarray(p_max, dtype=float).reshape(-1)
            assert pmax.shape == (n,)
            m.addConstr(p <= pmax)

    # Helper linear expression: Sigma @ y
    # (Gurobi likes explicit matrix*var expressions)
    Sig_y = m.addMVar(n, lb=-GRB.INFINITY, name="Sig_y")
    # Sig_y == Sigma y
    for i in range(n):
        m.addConstr(Sig_y[i] == gp.LinExpr(np.asarray(Sigma[i, :]).tolist(), y))

    # Primal feasibility
    # 1) y^T Σ y ≤ γ
    quad_risk = gp.QuadExpr()
    for i in range(n):
        for j in range(n):
            if Sigma[i, j] != 0.0:
                quad_risk.add(y[i] * Sigma[i, j] * y[j])
    m.addQConstr(quad_risk <= float(gamma), name="risk")

    # 2) 1^T y ≤ 1
    m.addConstr(gp.quicksum(y) <= 1.0, name="budget")

    # Stationarity: (c - p) - 2*lambda*(Sigma y) - mu*1 - nu = 0  (componentwise)
    # Rearranged to affine + bilinear == 0
    for i in range(n):
        # (c_i - p_i) - mu - nu_i - 2 * lambda * (Sigma y)_i == 0
        m.addQConstr((c[i] - p[i]) - mu - nu[i] - 2.0 * lam * Sig_y[i] == 0.0, name=f"stat_{i}")

    # Complementary slackness:
    # lambda * (y^T Σ y - γ) = 0
    m.addQConstr(lam * (quad_risk - float(gamma)) == 0.0, name="cs_risk")
    # mu * (1^T y - 1) = 0
    m.addQConstr(mu * (gp.quicksum(y) - 1.0) == 0.0, name="cs_budget")
    # nu_i * y_i = 0, for all i
    for i in range(n):
        m.addQConstr(nu[i] * y[i] == 0.0, name=f"cs_y_{i}")

    # Objective: maximize p^T y
    m.setObjective(p @ y, GRB.MAXIMIZE)

    m.optimize()

    status = m.Status
    sol = {
        "status": gp.GRB.Status.getname(status),
        "obj": None, "p": None, "y": None, "lambda": None, "mu": None, "nu": None
    }
    if status in [GRB.OPTIMAL, GRB.SUBOPTIMAL, GRB.TIME_LIMIT]:
        try:
            sol.update({
                "obj": m.ObjVal if m.SolCount > 0 else None,
                "p": np.array(p.X) if p.X else None,
                "y": np.array(y.X) if y.X else None,
                "lambda": lam.X if lam.X is not None else None,
                "mu": mu.X if mu.X is not None else None,
                "nu": np.array(nu.X) if nu.X else None,
            })
        except gp.GurobiError:
            pass

    return sol


In [ ]:
# ---------- Minimal example ----------
if __name__ == "__main__":
    np.random.seed(0)
    n = 6
    A = np.random.randn(n, n)
    Sigma = A @ A.T + 0.1 * np.eye(n)   # PSD
    c = np.random.uniform(0.02, 0.15, size=n)
    gamma = 0.5

    res = solve_bilevel_markowitz(c, Sigma, gamma, p_max=1.0, time_limit=60, verbose=True)
    print("Status:", res["status"])
    print("Objective (revenue):", res["obj"])
    print("p*:", np.round(res["p"], 4) if res["p"] is not None else None)
    print("y*:", np.round(res["y"], 4) if res["y"] is not None else None)

In [ ]:
a = np.asarray([1, 2, 3]).tolist()